# ACF Comparison — 4 Datasets (Channel 1)
Computes and overlays ACF curves for sfGFP, GFPuv, GFP_ex (end), GFP_sx (start) on a single plot.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
current_dir = Path().resolve()
from microlive.imports import *
from microlive import microscopy as mi
from pipeline_time_courses import compute_autocorrelation_for_dataset

In [ ]:
# ── Dataset Paths ─────────────────────────────────────────────────────────────
data_folder_sf        = Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/sfGFP/results')
data_folder_uv        = Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/GFPuv/results')
data_folder_end_xbp1  = Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/pRS038/results')
data_folder_start_xbp1= Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/pRS048/results')

list_datasets = [data_folder_sf, data_folder_uv, data_folder_end_xbp1, data_folder_start_xbp1]
list_names    = ['sfGFP', 'GFPuv', 'GFP_ex', 'GFP_sx']

# ── Colour palette (one per dataset) ─────────────────────────────────────────
list_colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']  # blue, green, orange, purple

In [ ]:
# ── Shared ACF Parameters ─────────────────────────────────────────────────────
step_size_in_sec                  = 5
start_lag                         = 1
channel_index                     = 1          # channel 1 for all datasets
selected_field                    = 'spot_int_ch_'
min_percentage_data_in_trajectory = 0.25
max_missing_frames                = 2
downsample                        = False
downsampling_factor               = 3
control_spots_mode                = False
use_global_mean                   = False
MAD_THRESHOLD_FACTOR              = 6
multi_tau_raw_points              = 60
multi_tau_bins_per_stage          = 16
min_snr                           = 0.5
smooth_window                     = 1
remove_outliers                   = True
correct_baseline                  = True
multi_tau                         = True
max_lag                           = 200
x_axes_min_max_list_values        = [-10, 1000]
y_axes_min_max_list_values        = [-0.02, 0.04]
fit_type                          = 'exponential'
de_correlation_threshold          = 0.001
use_linear_projection_for_lag_0   = True
gene_length                       = 1826       # codons (Xbp1u)

In [ ]:
# ── Run ACF for each dataset (suppress individual plots while running) ────────
all_results = []

for data_folder, name in zip(list_datasets, list_names):
    results_folder = data_folder / 'processing'
    results_folder.mkdir(exist_ok=True)
    plot_name = f'{name}_ACF.svg'
    print(f'Running ACF for {name} ...')
    r = compute_autocorrelation_for_dataset(
        dataset                          = 'cof',
        data_folder                      = data_folder,
        results_folder                   = results_folder,
        selected_field                   = selected_field,
        channel_index                    = channel_index,
        step_size_in_sec                 = step_size_in_sec,
        start_lag                        = start_lag,
        min_percentage_data_in_trajectory= min_percentage_data_in_trajectory,
        max_missing_frames               = max_missing_frames,
        downsample                       = downsample,
        downsampling_factor              = downsampling_factor,
        use_global_mean                  = use_global_mean,
        control_spots_mode               = control_spots_mode,
        correct_baseline                 = correct_baseline,
        min_snr                          = min_snr,
        smooth_window                    = smooth_window,
        remove_outliers                  = remove_outliers,
        MAD_THRESHOLD_FACTOR             = MAD_THRESHOLD_FACTOR,
        multi_tau                        = multi_tau,
        multi_tau_raw_points             = multi_tau_raw_points,
        multi_tau_bins_per_stage         = multi_tau_bins_per_stage,
        x_axes_min_max_list_values       = x_axes_min_max_list_values,
        max_lag                          = max_lag,
        index_max_lag_for_fit            = None,
        fit_type                         = fit_type,
        de_correlation_threshold         = de_correlation_threshold,
        use_linear_projection_for_lag_0  = use_linear_projection_for_lag_0,
        verbose                          = False,
        simulation_mode                  = False,
        SSA_data                         = None,
        line_color                       = (0.5, 0.5, 0.5),
        line_color_fit                   = 'dimgray',
        plot_name                        = plot_name,
        save_plots                       = False,
        figsize                          = (3.2, 2.2),
    )
    all_results.append(r)
    dwell_time = r['dwell_time']
    ke = gene_length / dwell_time
    ki = 1.0 / (r['mean_correlation'][1] * dwell_time)
    print(f'  -> dwell_time={dwell_time:.1f}s  ke={ke:.2f} cod/s  ki={ki:.4f} rib/s')

print('\nAll datasets done.')

In [ ]:
# ── Kinetics Summary Table ────────────────────────────────────────────────────
print(f"{'Dataset':<12}  {'dwell(s)':<10}  {'ki (rib/s)':<12}  {'ke (cod/s)':<12}")
print('-' * 52)
for r, name in zip(all_results, list_names):
    dwell_time = r['dwell_time']
    ke = gene_length / dwell_time
    ki = 1.0 / (r['mean_correlation'][1] * dwell_time)
    print(f"{name:<12}  {dwell_time:<10.1f}  {ki:<12.4f}  {ke:<12.2f}")

In [ ]:
# ── Overlay ACF Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

for r, name, color in zip(all_results, list_names, list_colors):
    lags        = np.array(r['lags'])
    mean_corr   = np.array(r['mean_correlation'])
    std_corr    = np.array(r['std_correlation'])
    dwell_time  = r['dwell_time']
    n_cells     = r['number_of_cells_final']   # post-filter: cells with ≥1 surviving trajectory
    n_traces    = r['number_of_trajectories_final']  # post-filter: trajectories after MAD removal

    x_min, x_max = x_axes_min_max_list_values
    mask = (lags > 0) & (lags <= x_max)

    ax.plot(lags[mask], mean_corr[mask],
            color=color, linewidth=1.5,
            label=f"{name}  τ={dwell_time:.0f}s  |  {n_cells} cells  {n_traces} traces")
    ax.fill_between(lags[mask],
                    mean_corr[mask] - std_corr[mask],
                    mean_corr[mask] + std_corr[mask],
                    color=color, alpha=0.15)

# Decorations
ax.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.5)
ax.set_xlabel('τ (s)', fontsize=11)
ax.set_ylabel('G(τ)', fontsize=11)
ax.set_title('ACF Comparison — Channel 1', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, framealpha=0.9)
ax.set_xlim(x_axes_min_max_list_values)
ax.set_xlim(0, 800)
ax.set_ylim(-0.02, 0.06)
ax.grid(False)
plt.tight_layout()
plt.savefig('ACF_comparison_ch1.svg', dpi=300, bbox_inches='tight')
plt.savefig('ACF_comparison_ch1.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ACF_comparison_ch1.svg / .png')